# Multithreading
Le multithreading peut être utilisé pour parralléliser des tâches `I/O bounds` ce sont des process dont le temps d'attente est dû a l'attente d'input ou d'output.  
Il  y a plusieur manière de faire du code asynchrone en python. 

## Sources
Les différentes sources pour cette fiche :  

https://realpython.com/intro-to-python-threading/  
https://realpython.com/python-concurrency/  

### threading et ThreadPoolEexcutor

https://www.youtube.com/watch?v=IEEhzQoKtQU

https://github.com/CoreyMSchafer/code_snippets/tree/master/Python/Threading

### asyncio
https://www.youtube.com/watch?v=Qb9s3UiMSTA

## Exemple synchrone

Avant tout un exemple simple qu'on retravaillera pour faire de l'asynchrone


In [1]:
import time

start = time.perf_counter()

def do_something(seconds):
    print(f'Sleeping {seconds} second(s)...')
    time.sleep(seconds)
    print(f'Done Sleeping...{seconds}')

do_something(1)
do_something(1)

finish = time.perf_counter()

print(f'Finished in {round(finish-start, 2)} second(s)')

Sleeping 1 second(s)...
Done Sleeping...1
Sleeping 1 second(s)...
Done Sleeping...1
Finished in 2.0 second(s)


## module threading

### Lancement de plusieurs tâches en parralééle

Dans cet exemple on peut voir que les tâches sont bien lancée de manière asynchrone en parralléle mais l'execution du script se termine avant que les tâches ne soit terminée.   

Pour attendre la fin des tâches il faut utiiser la method join.


In [2]:
import time
import threading

start = time.perf_counter()

def do_something(seconds):
    print(f'Sleeping {seconds} second(s)...')
    time.sleep(seconds)
    print(f'Done Sleeping...{seconds}')

for _ in range(10):
    t = threading.Thread(target=do_something, args=[1.5])
    t.start()

finish = time.perf_counter()

print(f'Finished in {round(finish-start, 2)} second(s)')

Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Finished in 0.01 second(s)
Done Sleeping...1.5Done Sleeping...1.5
Done Sleeping...1.5

Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5



### Attente des résultats avec join

Ici on a bien un temps d'execution qui est cohérent avec l'execution en parralléle


In [3]:
import time
import threading

start = time.perf_counter()

def do_something(seconds):
    print(f'Sleeping {seconds} second(s)...')
    time.sleep(seconds)
    print(f'Done Sleeping...{seconds}')

threads = []

for _ in range(10):
    t = threading.Thread(target=do_something, args=[1.5])
    t.start()
    threads.append(t)

for thread in threads:
    thread.join()
    
finish = time.perf_counter()

print(f'Finished in {round(finish-start, 2)} second(s)')

Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Finished in 1.51 second(s)


## ThreadPoolExecutor

### method submit

Un exemple utilisant le constructeur `ThreadPoolExecutor` avec récupération du résultat


In [4]:
import time
import concurrent.futures

start = time.perf_counter()

def do_something(seconds):
    print(f'Sleeping {seconds} second(s)...')
    time.sleep(seconds)
    return f'Done Sleeping...{seconds}'

results = []

with concurrent.futures.ThreadPoolExecutor() as executor:
    for _ in range(10):
        results.append( executor.submit(do_something, 1.5) )
    
for result in results:
    print(result.result())
    
finish = time.perf_counter()

print(f'Finished in {round(finish-start, 2)} second(s)')

Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Sleeping 1.5 second(s)...
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Done Sleeping...1.5
Finished in 3.01 second(s)



### method map

La methode map permet de simplifier la créatin de la liste de résultat. 

on peut constater que l'ordre de création est conservé.


In [5]:
import concurrent.futures
import time

start = time.perf_counter()


def do_something(seconds):
    print(f'Sleeping {seconds} second(s)...')
    time.sleep(seconds)
    return f'Done Sleeping...{seconds}'


with concurrent.futures.ThreadPoolExecutor() as executor:
    secs = [5, 4, 3, 2, 1]
    results = executor.map(do_something, secs)

for result in results:
    print(result)

finish = time.perf_counter()

print(f'Finished in {round(finish-start, 2)} second(s)')

Sleeping 5 second(s)...
Sleeping 4 second(s)...
Sleeping 3 second(s)...
Sleeping 2 second(s)...
Sleeping 1 second(s)...
Done Sleeping...5
Done Sleeping...4
Done Sleeping...3
Done Sleeping...2
Done Sleeping...1
Finished in 5.01 second(s)


# Asyncio module

Cet exemple ne fonctionne pas dans le notebook et doit être executé en python 3.7 au minimum.  
Interessant a creuser pour mettre des lock semaphore etc...

```python
# https://www.youtube.com/watch?v=Qb9s3UiMSTA

import time, asyncio

async def do_something(seconds):
    print(f'Sleeping {seconds} second(s)...')
    await asyncio.sleep(seconds)
    print(f'Done Sleeping...{seconds}')
    return f'Result Sleeping...{seconds}'

async def main():
    batch = asyncio.gather( *[ do_something(sec) for sec in range(1,6) ]  )
    results = await batch
    return results

if __name__ == '__main__':
    start = time.perf_counter()
    results = asyncio.run( main() )
    for result  in results:
        print( result )
    finish = time.perf_counter()
    print(f'Finished in {round(finish-start, 2)} second(s)')
```